<a href="https://colab.research.google.com/github/Sanhith30/ITA0605-ML/blob/main/Practical_ProblemsML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practical Problems: FIND-S and Candidate-Elimination Algorithms

This notebook demonstrates the implementation of:
1. **FIND-S Algorithm** to find the most specific hypothesis.
2. **Candidate-Elimination Algorithm** to output the version space (most specific and most general hypotheses) consistent with the training examples.

In [5]:
import pandas as pd
import numpy as np

csv_path = '/content/enjoysport - enjoysport.csv'

print(f"Loading dataset from: {csv_path}")
df = pd.read_csv(csv_path)
display(df)

Loading dataset from: /content/enjoysport - enjoysport.csv


,sky,airtemp,humidity,wind,water,forcast,enjoysport
0,sunny,warm,normal,strong,warm,same,yes
1,sunny,warm,high,strong,warm,same,yes
2,rainy,cold,high,strong,warm,change,no
3,sunny,warm,high,strong,cool,change,yes


## Practical Problem 1: FIND-S Algorithm

Find-S finds the most specific hypothesis that fits all the positive training examples.

In [6]:
def find_s(concepts, target):
    # Initialize the most specific hypothesis with the first positive example
    for i, val in enumerate(target):
        if val.lower() in ['yes', '1', 'true']:
            specific_h = concepts[i].copy()
            break

    print("Initialization of specific_h:", specific_h)

    for i, val in enumerate(concepts):
        if target[i].lower() in ['yes', '1', 'true']:
            for x in range(len(specific_h)):
                if val[x] != specific_h[x]:
                    specific_h[x] = '?'
            print(f"After sample {i+1}: {specific_h}")

    return specific_h

# Extract concepts and target
concepts = np.array(df.iloc[:, :-1])
target = np.array(df.iloc[:, -1])

print("Applying FIND-S Algorithm:")
final_hypothesis = find_s(concepts, target)
print("\nFinal Most Specific Hypothesis:", final_hypothesis)

Applying FIND-S Algorithm:
Initialization of specific_h: ['sunny' 'warm' 'normal' 'strong' 'warm' 'same']
After sample 1: ['sunny' 'warm' 'normal' 'strong' 'warm' 'same']
After sample 2: ['sunny' 'warm' '?' 'strong' 'warm' 'same']
After sample 4: ['sunny' 'warm' '?' 'strong' '?' '?']

Final Most Specific Hypothesis: ['sunny' 'warm' '?' 'strong' '?' '?']


## Practical Problem 2: Candidate-Elimination Algorithm

Candidate-Elimination computes the version space containing all hypotheses consistent with the training examples.

In [7]:
def candidate_elimination(concepts, target):
    num_attributes = len(concepts[0])

    # Initialize Specific and General Boundaries
    specific_h = ["0"] * num_attributes
    general_h = [["?"] * num_attributes for _ in range(num_attributes)]

    # Find first positive example to initialize specific hypothesis
    for i, val in enumerate(target):
        if val.lower() in ['yes', '1', 'true']:
            specific_h = concepts[i].copy()
            break

    general_h = [["?"] * num_attributes for _ in range(num_attributes)]

    print(f"Initialization:")
    print(f"S0: {specific_h}")
    print(f"G0: {general_h}\n")

    for i, h in enumerate(concepts):
        if target[i].lower() in ['yes', '1', 'true']:
            # Update Specific Boundary for Positive Example
            for x in range(num_attributes):
                if h[x] != specific_h[x]:
                    specific_h[x] = '?'
                    # Filter general boundary consistent with new specific boundary
                    for g in general_h:
                        if g[x] != '?' and g[x] != specific_h[x]:
                            general_h.remove(g)
        else:
            # Update General Boundary for Negative Example
            for x in range(num_attributes):
                if h[x] != specific_h[x]:
                    # Create a specialized hypothesis
                    g_temp = ['?'] * num_attributes
                    g_temp[x] = specific_h[x]
                    if g_temp not in general_h:
                        general_h.append(g_temp)
                else:
                    # Remove incompatible general hypotheses
                    for g in list(general_h):
                        if g[x] == h[x]:
                            general_h.remove(g)

        # Remove any purely generic default placeholders or redundant entries
        general_h = [g for g in general_h if g != ['?'] * num_attributes]

        print(f"Iteration {i+1} (Sample {i+1} is {target[i]}):")
        print(f"S{i+1}: {specific_h}")
        print(f"G{i+1}: {general_h}\n")

    return specific_h, general_h

s_final, g_final = candidate_elimination(concepts, target)
print("Final Specific Boundary (S):", s_final)
print("Final General Boundary (G):", g_final)

Initialization:
S0: ['sunny' 'warm' 'normal' 'strong' 'warm' 'same']
G0: [['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?']]

Iteration 1 (Sample 1 is yes):
S1: ['sunny' 'warm' 'normal' 'strong' 'warm' 'same']
G1: []

Iteration 2 (Sample 2 is yes):
S2: ['sunny' 'warm' '?' 'strong' 'warm' 'same']
G2: []

Iteration 3 (Sample 3 is no):
S3: ['sunny' 'warm' '?' 'strong' 'warm' 'same']
G3: [['sunny', '?', '?', '?', '?', '?'], ['?', 'warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', 'same']]

Iteration 4 (Sample 4 is yes):
S4: ['sunny' 'warm' '?' 'strong' '?' '?']
G4: [['sunny', '?', '?', '?', '?', '?'], ['?', 'warm', '?', '?', '?', '?']]

Final Specific Boundary (S): ['sunny' 'warm' '?' 'strong' '?' '?']
Final General Boundary (G): [['sunny', '?', '?', '?', '?', '?'], ['?', 'warm', '?', '?', '?', '?']]


## Practical Problem 3: Decision Tree based ID3 Algorithm

This section demonstrates the implementation of the ID3 (Iterative Dichotomiser 3) algorithm to construct a decision tree and classify a new test sample.

In [8]:
import numpy as np
import pandas as pd

# Create a standard PlayTennis dataset for ID3 demonstration
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rainy', 'Rainy', 'Rainy', 'Overcast', 'Sunny', 'Sunny', 'Rainy', 'Sunny', 'Overcast', 'Overcast', 'Rainy'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}

play_df = pd.DataFrame(data)
print("Dataset for ID3 Decision Tree:")
display(play_df)

Dataset for ID3 Decision Tree:


,Outlook,Temperature,Humidity,Wind,PlayTennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rainy,Mild,High,Weak,Yes
4,Rainy,Cool,Normal,Weak,Yes
5,Rainy,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rainy,Mild,Normal,Weak,Yes


In [9]:
def entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    entropy_val = np.sum([(-counts[i]/np.sum(counts)) * np.log2(counts[i]/np.sum(counts)) for i in range(len(elements))])
    return entropy_val

def info_gain(data, split_attribute_name, target_name="PlayTennis"):
    total_entropy = entropy(data[target_name])
    vals, counts = np.unique(data[split_attribute_name], return_counts=True)

    weighted_entropy = np.sum([(counts[i]/np.sum(counts)) * entropy(data[data[split_attribute_name] == vals[i]][target_name]) for i in range(len(vals))])
    information_gain = total_entropy - weighted_entropy
    return information_gain

def id3(data, original_data, features, target_attribute_name="PlayTennis", parent_node_class=None):
    # If all target values are the same, return this value
    if len(np.unique(data[target_attribute_name])) <= 1:
        return np.unique(data[target_attribute_name])[0]

    # If dataset is empty, return the mode target value of original data
    elif len(data) == 0:
        return np.unique(original_data[target_attribute_name])[np.argmax(np.unique(original_data[target_attribute_name], return_counts=True)[1])]

    # If feature list is empty, return parent node class
    elif len(features) == 0:
        return parent_node_class

    else:
        # Determine default node class for child nodes
        parent_node_class = np.unique(data[target_attribute_name])[np.argmax(np.unique(data[target_attribute_name], return_counts=True)[1])]

        # Select the feature that best splits the dataset
        item_values = [info_gain(data, feature, target_attribute_name) for feature in features]
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]

        # Create tree structure
        tree = {best_feature: {}}
        features = [i for i in features if i != best_feature]

        # Grow branches
        for value in np.unique(original_data[best_feature]):
            sub_data = data[data[best_feature] == value]
            subtree = id3(sub_data, original_data, features, target_attribute_name, parent_node_class)
            tree[best_feature][value] = subtree

        return tree

features = [col for col in play_df.columns if col != 'PlayTennis']
tree = id3(play_df, play_df, features)

print("Generated ID3 Decision Tree structure:")
import pprint
pprint.pprint(tree)

Generated ID3 Decision Tree structure:
{'Outlook': {'Overcast': 'Yes',
             'Rainy': {'Wind': {'Strong': 'No', 'Weak': 'Yes'}},
             'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}}}


In [10]:
def predict(query, tree, default='Yes'):
    for key in list(query.keys()):
        if key in list(tree.keys()):
            try:
                result = tree[key][query[key]]
            except KeyError:
                return default

            if isinstance(result, dict):
                return predict(query, result)
            else:
                return result

# Classify a new sample
new_sample = {'Outlook': 'Rainy', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Strong'}
prediction = predict(new_sample, tree)

print(f"New sample to classify: {new_sample}")
print(f"Prediction (PlayTennis): {prediction}")

New sample to classify: {'Outlook': 'Rainy', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Strong'}
Prediction (PlayTennis): No
